#### **Setup & Data Loading**

In [1]:
import pandas as pd
import numpy as np
import random

# Load datasets
DATA_PATH = "../data/raw"
item_master = pd.read_csv(f"{DATA_PATH}/d365_item_master.csv")
sales_lines = pd.read_csv(f"{DATA_PATH}/d365_sales_lines.csv")

print("-- Master Data Loaded Successfully --")
print(f"Item Master Records: {len(item_master)}")
print(f"Sales Lines Records: {len(sales_lines)}")
item_master.head(3)

-- Master Data Loaded Successfully --
Item Master Records: 180
Sales Lines Records: 1000


,No_,Description,Brand,Item_Category_Code,HS_Code,Net_Weight_KG,Unit_Volume_M3,Base_Cost_EUR,Unit_Price_AED,Gen_Prod_Posting_Group,Created_Date
0,SKU-THO-1001,Thomas Pink Tailored Suits & Blazers Item #1,Thomas Pink,Tailored Suits & Blazers,6403.59.00,1.13,0.0014,98.17,1212.48,FINISHED_GOODS,2025-04-20
1,SKU-THO-1002,Thomas Pink Tailored Suits & Blazers Item #2,Thomas Pink,Tailored Suits & Blazers,6105.10.00,1.03,0.0021,62.47,629.39,NaN,2025-02-27
2,SKU-LUX-1003,LuxFootwear Milano Fine Leather Footwear Item #3,LuxFootwear Milano,Fine Leather Footwear,6105.10.00,3.08,0.0041,248.63,2587.51,FINISHED_GOODS,2025-03-29


#### **SKU Pre-Distribution Readiness Scoring Engine**

In [9]:
def score_sku_readiness(row):
    score = 100
    missing_fields = []
    
    if pd.isna(row['HS_Code']) or str(row['HS_Code']).strip() == "":
        score -= 30
        missing_fields.append('HS_Code')
    if pd.isna(row['Gen_Prod_Posting_Group']) or str(row['Gen_Prod_Posting_Group']).strip() == "":
        score -= 25
        missing_fields.append('Gen_Prod_Posting_Group')
    if row['Net_Weight_KG'] <= 0:
        score -= 20
        missing_fields.append('Net_Weight_KG')
    if row['Unit_Volume_M3'] <= 0:
        score -= 15
        missing_fields.append('Unit_Volume_M3')
    if row['Unit_Price_AED'] <= 0:
        score -= 10
        missing_fields.append('Unit_Price_AED')
        
    return pd.Series({
        'Readiness_Score': max(score, 0),
        'Is_Distribution_Ready': score >= 85,
        'Missing_Attributes': ", ".join(missing_fields) if missing_fields else "None"
    })

# Apply scoring to item master
sku_audit_df = item_master.copy()
sku_audit_df[['Readiness_Score', 'Is_Distribution_Ready', 'Missing_Attributes']] = sku_audit_df.apply(score_sku_readiness, axis=1)

# Summary metrics
ready_count = sku_audit_df['Is_Distribution_Ready'].sum()
blocked_count = len(sku_audit_df) - ready_count

print("-- SPORTLUX SKU READINESS SCORECARD --")
print("========================================")
print(f"Total SKUs Audited          : {len(sku_audit_df)}")
print(f"Ready for Distribution      : {ready_count}")
print(f"Blocked (Requires Correction) : {blocked_count}")
print("==========================================")
sku_audit_df[['No_', 'Description', 'Readiness_Score', 'Is_Distribution_Ready', 'Missing_Attributes']].head(5)

-- SPORTLUX SKU READINESS SCORECARD --
Total SKUs Audited          : 180
Ready for Distribution      : 112
Blocked (Requires Correction) : 68


,No_,Description,Readiness_Score,Is_Distribution_Ready,Missing_Attributes
0,SKU-THO-1001,Thomas Pink Tailored Suits & Blazers Item #1,100,True,None
1,SKU-THO-1002,Thomas Pink Tailored Suits & Blazers Item #2,75,False,Gen_Prod_Posting_Group
2,SKU-LUX-1003,LuxFootwear Milano Fine Leather Footwear Item #3,100,True,None
3,SKU-THO-1004,Thomas Pink Tailored Suits & Blazers Item #4,75,False,Gen_Prod_Posting_Group
4,SKU-THO-1005,Thomas Pink Luxury Outerwear Item #5,100,True,None


#### **B2B Customer Profile Onboarding Compliance Audit**

In [6]:
b2b_destinations = sales_lines[sales_lines['Sales_Channel'] == 'B2B Wholesale']['Destination'].unique()

b2b_customers = []
for idx, dest in enumerate(b2b_destinations):
    has_trn = (idx % 3 != 0)  # Inject missing TRN compliance gaps
    b2b_customers.append({
        "Customer_No": f"CUST-B2B-{201 + idx}",
        "Customer_Name": dest,
        "Tax_Registration_No": f"TRN-UAE-{random.randint(100000, 999999)}" if has_trn else None,
        "Credit_Limit_AED": 500000.0,
        "Payment_Terms": "NET-30"
    })

b2b_df = pd.DataFrame(b2b_customers)

# Score customer readiness
b2b_df['Tax_Compliance_Verified'] = b2b_df['Tax_Registration_No'].notnull()
b2b_df['Onboarding_Status'] = b2b_df['Tax_Compliance_Verified'].apply(
    lambda x: 'Approved' if x else 'Blocked (Missing UAE TRN)'
)

print("-- B2B Customer Compliance Audit --")
display(b2b_df)

-- B2B Customer Compliance Audit --


,Customer_No,Customer_Name,Tax_Registration_No,Credit_Limit_AED,Payment_Terms,Tax_Compliance_Verified,Onboarding_Status
0,CUST-B2B-201,Dubai Mall Flagship,None,500000.0,NET-30,False,Blocked (Missing UAE TRN)
1,CUST-B2B-202,Mall of the Emirates,TRN-UAE-346695,500000.0,NET-30,True,Approved
2,CUST-B2B-203,D2C UAE Delivery,TRN-UAE-198313,500000.0,NET-30,True,Approved
3,CUST-B2B-204,Riyadh B2B Hub,None,500000.0,NET-30,False,Blocked (Missing UAE TRN)
4,CUST-B2B-205,Harvey Nichols Dubai,TRN-UAE-774288,500000.0,NET-30,True,Approved


#### **Exporting Gatekeeper Exception Summary**

In [5]:
blocked_skus = sku_audit_df[~sku_audit_df['Is_Distribution_Ready']]
blocked_customers = b2b_df[b2b_df['Onboarding_Status'].str.startswith('Blocked')]

print(f"Pre-Distribution Gatekeeper Alert:")
print(f"   - {len(blocked_skus)} SKUs are blocked from warehouse picking due to missing master data attributes.")
print(f"   - {len(blocked_customers)} B2B wholesale accounts require compliance document sign-off.")

Pre-Distribution Gatekeeper Alert:
   - 68 SKUs are blocked from warehouse picking due to missing master data attributes.
   - 2 B2B wholesale accounts require compliance document sign-off.
